In [1]:
from __future__ import annotations
import os
import json
import importlib
import argparse
from functools import partial
from collections import defaultdict

import random
import numpy as np
import pickle

import torch

from vllm import LLM, SamplingParams, PoolingParams

from sal.config import Config

from core import mcts_search_mini_v15
from core import mcts_search_mini_v55
from core import mcts_search_mini_v35
from core import mcts_search_mini_v45
from core.reward_models import RLHFFlow
from utils.load_data import load_data_prm800k

# from core.llm_engine import rm_engine
# from core.llms import rm_generate
# import logging
# logging.basicConfig(format='%(message)s', level=logging.ERROR)

INFO 09-04 08:30:03 [__init__.py:244] Automatically detected platform cuda.


In [2]:
if torch.cuda.is_available():
    GPUS = os.environ.get('CUDA_VISIBLE_DEVICES', "0").split(',')
    print(GPUS)
else:
    print("CUDA is not available.")

['0']


In [3]:
# base_dir
base_dir = '/groups/kjun/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-Modified"

In [4]:
stop

NameError: name 'stop' is not defined

In [5]:
#  load data 
data_by_levels = load_data_prm800k(data_dir)

1: 43
2: 90
3: 105
4: 128
5: 134


In [6]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# general params
config = Config()
config.agg_strategy = 'last'

config.n = 4                      # number of budgets to be generated per depth
config.beam_width = 4             # number of nodes left after selection
config.lookahead = 0              # don't use it for now
config.max_depths = 7            # max depths, after reaching max_depth then terminate search 
config.sort_completed = False      
config.filter_duplicates = True   # remove any duplicates in the last list of trajs
config.seed = 0

# mcts parameter
config.num_phases = 1
config.num_batches = 1
config.batch_budget = config.num_batches*config.max_depths 

config.lam = 10 
config.normalize_embeds = True

config.cpuct_root = 0
config.cpuct_leaf = 0
config.cpuct = 2
config.ds_beta = 1.0
config.ds_alpha = 10.0
config.use_ppl = True

config.version = "v51"



In [34]:
level = 4                                   # level of difficulty of questions
num_questions = len(data_by_levels[level])  # level 4 has 128 questions
# num_questions = 1
print(f"num_questions = {num_questions}")

# get batch of questions ['q1', 'q2', ...]
# batch_of_questions = [data_by_levels[level][q_idx]['problem'] for q_idx in range(num_questions)]
q_idx = 0
batch_of_questions = [data_by_levels[level][q_idx]['problem']]
batch_of_answers = [data_by_levels[level][q_idx]['answer']]
print(batch_of_questions)
print(batch_of_answers)

num_questions = 128
['The set of points $(x,y,z)$ that satisfy\n\\[2x = 3y = -z\\]is a line.\n\nThe set of points $(x,y,z)$ that satisfy\n\\[6x = -y = -4z\\]is another line.\n\nFind the angle between these lines, in degrees.']
['90^\\circ']


In [35]:
trial_idx = 0
config.max_depths = 7
config_name = f"mcts--n-{config.n}--d-{config.max_depths}--normalize-{config.normalize_embeds}--level-{level}--qidx-{q_idx}--trial-{trial_idx}"
print(config_name)
with open(f"results/tree_{config_name}.pkl", 'rb') as fout:
    tree_dict = pickle.load(fout)

mcts--n-4--d-7--normalize-True--level-4--qidx-0--trial-0


In [ ]:

completions_dict_base = defaultdict()
for tag, output in tree_dict.items():
    if output["is_completed"]:
        completions_dict_base[tag] = output["text"]

# config_name = f"mcts--base--n-{config.n}--d-{config.max_depths}--level-{level}--qidx-{q_idx}"
# with open(f"results/sample_{config_name}--trial-{trial_idx}.jsonl", 'w', encoding = 'utf-8') as fout:
#     json.dump(completions_dict_base, fout)
#     fout.write('\n')

In [ ]:
# print(tree_dict['0.0.0']['text'])
# print(tree_dict['0.0.1']['text'])
# print(tree_dict['0.0.2']['text'])
# print(tree_dict['0.0.3']['text'])
print(tree_dict['0.0.0.2.1.0.1.1'])
print(tree_dict['0.0.0.2.1.0.1.2'])
print(tree_dict['0.0.0.0.0.0.2'])
print(tree_dict['0.0.0.0.0.0.1.0'])

In [30]:
config.n = 4
config.max_depths = 7
# config.ds_alpha = 100.0 
config.ds_beta = 1.0
config.ds_alpha = 100.0
config.diversity_thres = 3
config.lam = 10
config.cpuct = 2

# config.num_phases = config.n**(config.max_depths-1)
config.num_batches = 2
config.batch_budget = config.num_batches*config.max_depths
config.num_phases = config.batch_budget
print(config.batch_budget)
importlib.reload(mcts_search_mini_v55)
importlib.reload(mcts_search_mini_v15)
importlib.reload(mcts_search_mini_v35)
# importlib.reload(mcts_search_mini_v45)
# importlib.reload(mcts_search_mini_v31)
# importlib.reload(mcts_search_mini_v41)

14


<module 'core.mcts_search_mini_v35' from '/home/u20/tnguyen9210/tnn1/LLMs/llm-reasoning-methods/core/mcts_search_mini_v35.py'>

In [ ]:

for question in batch_of_questions:
    agent = mcts_search_mini_v35.MCTS(config=config, question=question)
    tree_dict_v35, tag_list_v35, completions_dict_v35 = mcts_search_mini_v35.mcts_search(question, agent, config, tree_dict)
    # print(agent_completions)

config_name = f"mcts--v35--n-{config.n}--d-{config.max_depths}--nb-{config.num_batches}--lam-{config.lam}--dalpha-{config.ds_alpha}--dbeta-{config.ds_beta}--cpuct-{config.cpuct_root}-{config.cpuct_leaf}-{config.cpuct}--level-{level}--qidx-{q_idx}"

# with open(f"results/sample_{config_name}--trial-{trial_idx}.jsonl", 'w', encoding = 'utf-8') as fout:
#     json.dump(completions_dict_v35, fout)
#     fout.write('\n')

In [ ]:
tags_list_v35 = set(completions_dict_v35.keys())
print(tags_list_v35)
print(f"num_tags = {len(completions_dict_v35)}")
for tag, text in completions_dict_v35.items():
    print(f"-> tag = {tag}, q_value = {tree_dict[tag]['q_value']:0.4f}")
    print(text)


In [ ]:

for question in batch_of_questions:
    agent = mcts_search_mini_v55.MCTS(config=config, question=question)
    tree_dict_v55, tag_list_v55, completions_dict_v55 = mcts_search_mini_v55.mcts_search(question, agent, config, tree_dict)
    # print(agent_completions)

config_name = f"mcts--v55--n-{config.n}--d-{config.max_depths}--nb-{config.num_batches}--lam-{config.lam}--dalpha-{config.ds_alpha}--dbeta-{config.ds_beta}--cpuct-{config.cpuct_root}-{config.cpuct_leaf}-{config.cpuct}--level-{level}--qidx-{q_idx}"

with open(f"results/sample_{config_name}--trial-{trial_idx}.jsonl", 'w', encoding = 'utf-8') as fout:
    json.dump(completions_dict_v55, fout)
    fout.write('\n')

In [ ]:
tags_list_v55 = set(completions_dict_v55.keys())
print(tags_list_v55)
print(f"num_tags = {len(completions_dict_v55)}")
for tag, text in completions_dict_v55.items():
    print(f"-> tag = {tag}, q_value = {tree_dict[tag]['q_value']:0.4f}")
    # print(text)


In [38]:

importlib.reload(mcts_search_mini_v15)
for question in batch_of_questions:
    agent = mcts_search_mini_v15.MCTS(config=config, question=question)
    tree_dict_v15, tag_list_v15, completions_dict_v15 = mcts_search_mini_v15.mcts_search(question, agent, config, tree_dict)
    # print(agent_completions)

# config_name = f"mcts--v15--n-{config.n}--d-{config.max_depths}--nb-{config.num_batches}--cpuct-{config.cpuct_root}-{config.cpuct_leaf}-{config.cpuct}--level-{level}--qidx-{q_idx}"
# with open(f"results/sample_{config_name}--trial-{trial_idx}.jsonl", 'w', encoding = 'utf-8') as fout:
#     json.dump(completions_dict_v15, fout)
#     fout.write('\n')


-> p = 0

-> selection
from_root = True
start_node = 0
selected_node = 0

-> d = 0

-> selection
from_root = False
start_node = 0
0.0
   q-value = 0.1023
   u-value = 0.0000
   puct = 0.1023
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.1
   q-value = 0.2069
   u-value = 0.0000
   puct = 0.2069
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.2
   q-value = 0.1755
   u-value = 0.0000
   puct = 0.1755
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
0.3
   q-value = 0.4766
   u-value = 0.0000
   puct = 0.4766
   nvisit = 1.00
   parent.nvisit = 0.00
   is_terminal = False
selected_child = 0.3
selected_node = 0.3
bcnt used 1

-> d = 1

-> selection
from_root = False
start_node = 0.3
0.3.0
   q-value = 0.7432
   u-value = 0.0000
   puct = 0.7432
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.3.1
   q-value = 0.7520
   u-value = 0.0000
   puct = 0.7520
   nvisit = 1.00
   parent.nvisit = 1.00
   is_terminal = False
0.3

In [ ]:
tags_list_v15 = set(completions_dict_v15.keys())
print(tags_list_v15)
print(f"num_tags = {len(completions_dict_v15)}")
for tag, text in completions_dict_v15.items():
    print(f"tag = {tag}, q_value = {tree_dict[tag]['q_value']:0.4f}")
    # print(text)

In [ ]:
tags_common = tags_list_v15 & tags_list_v55
tags_diff_v15 = tags_list_v15 - tags_common
tags_diff_v55 = tags_list_v55 - tags_common
print(tags_common)
print(tags_diff_v15)
print(tags_diff_v55)

In [ ]:
tags_common = tags_list_v15 & tags_list_v35
tags_diff_v15 = tags_list_v15 - tags_common
tags_diff_v35 = tags_list_v35 - tags_common
print(tags_common)
print(tags_diff_v15)
print(tags_diff_v35)

In [ ]:
# print(tree_dict['0.1.3.3.0.3.2.0'])
# print(tree_dict['0.3.3.0.3.0.0.1'])
# print(tree_dict['0.0.2.2.2.3.0'])
print(tree_dict['0.0.3.1.0.1.0'])
# print(tree_dict['0.1']['text'])
# print(tree_dict['0.2']['text'])
# print(tree_dict['0.3']['text'])
# print(tree_dict['0.0.0.0.0.0.0.0.1'])
# print(tree_dict['0.0.0.0.0.0.0.0.2'])
print()
# print(tree_dict['0.1.3.2.1.0.1'])
# print(tree_dict['0.1.3.2.1.0.3'])